In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import gwaslab as gl

from scipy.stats import chi2
from qmplot import qqplot
from plots.plotting import (
    load_plink_glm,
    miami_plot_with_PAR,
    manhattan_plot_with_PAR,
    manhattan_plot_PAR_only,
    manhattan_plot_PAR_with_highlights,
)

In [ ]:
# Release of interest
rel = 12

In [ ]:
wd = '/YOUR/WORKING/DIR'

### Miami Plot of GWAS Results

In [ ]:
# Plot results from both sexes and regions
label = 'EUR'
maf_thresh = 0.03

region = 'non_PAR'
fem_nonpar_path = f'{wd}/working/R{rel}_prep/XWAS/female/{region}/XWAS_{label}_{region}_female.PHENO.glm.logistic.hybrid'
male_nonpar_path= f'{wd}/working/R{rel}_prep/XWAS/male/{region}/XWAS_{label}_{region}_male.PHENO.glm.logistic.hybrid'

region = 'PAR'
fem_par_path = f'{wd}/working/R{rel}_prep/XWAS/female/{region}/XWAS_{label}_{region}_female.PHENO.glm.logistic.hybrid'
male_par_path= f'{wd}/working/R{rel}_prep/XWAS/male/{region}/XWAS_{label}_{region}_male.PHENO.glm.logistic.hybrid'

df_f_nonpar = load_plink_glm(fem_nonpar_path)
df_f_par = load_plink_glm(fem_par_path)
df_m_nonpar = load_plink_glm(male_nonpar_path)
df_m_par = load_plink_glm(male_par_path)

df_f_nonpar = df_f_nonpar[df_f_nonpar.A1_FREQ >= maf_thresh]
df_f_par = df_f_par[df_f_par.A1_FREQ >= maf_thresh]
df_m_nonpar = df_m_nonpar[df_m_nonpar.A1_FREQ >= maf_thresh]
df_m_par = df_m_par[df_m_par.A1_FREQ >= maf_thresh]

fig = miami_plot_with_PAR(
    male_nonpar=df_m_nonpar,
    male_par=df_m_par,
    fem_nonpar=df_f_nonpar,
    fem_par=df_f_par,
    title=f"{label} X Chromosome Results",
    genomewide=5e-8,
    suggestive=1e-6,
    nonpar_color="#76b7e5",
    par_color="#4f77be",
    point_size=5,
    annotate_top_n=0
)

In [ ]:
fig.savefig(f"{wd}/{label}_R{rel}_miami_chrX_MAF.png", dpi=600, bbox_inches="tight")
fig.show()

### Manhattan Plot of Combined Results

In [ ]:
# Plot sex-combined results from both regions
region = 'non_PAR'
nonpar_path = f'{wd}/working/R{rel}_prep/XWAS/combined/{region}/XWAS_{label}_{region}_combined.PHENO.glm.logistic.hybrid'

region = 'PAR'
par_path = f'{wd}/working/R{rel}_prep/XWAS/combined/{region}/XWAS_{label}_{region}_combined.PHENO.glm.logistic.hybrid'

df_nonpar = load_plink_glm(nonpar_path)
df_par = load_plink_glm(par_path)

df_nonpar = df_nonpar[df_nonpar.A1_FREQ >= maf_thresh]
df_par = df_par[df_par.A1_FREQ >= maf_thresh]

fig = manhattan_plot_with_PAR(
    merged_nonpar=df_nonpar,
    merged_par=df_par,
    title=f"{label} X Chromosome Results",
    genomewide=5e-8,
    suggestive=1e-6,
    nonpar_color="#76b7e5",
    par_color="#4f77be",
    point_size=5,
    annotate_top_n=0
)

In [ ]:
fig.savefig(f"{wd}/{label}_R{rel}_manhattan_chrX_MAF.png", dpi=600, bbox_inches="tight")
fig.show()

### PAR1-Only Manhattan Plot

In [ ]:
# PAR2 is sparsely imputed, so plot PAR1 on its own
region = 'PAR'
par_path = f'{wd}/working/R{rel}_prep/XWAS/combined/{region}/XWAS_{label}_{region}_combined.PHENO.glm.logistic.hybrid'

df_par1 = load_plink_glm(par_path)
df_par1 = df_par1[df_par1.A1_FREQ >= maf_thresh]
df_par1 = df_par1[df_par1.SNP.str.contains('PAR1')]

fig = manhattan_plot_PAR_only(
    merged_par=df_par1,
    title=f"{label} X Chromosome PAR1 Results",
    genomewide=5e-8,
    suggestive=1e-6,
    par_color="#4f77be",
    point_size=5,
    annotate_top_n=0
)

In [ ]:
fig.savefig(f"{wd}/{label}_R{rel}_manhattan_PAR1_MAF.png", dpi=600, bbox_inches="tight")
fig.show()

In [ ]:
# Highlight the two most significant PAR1 SNPs
green_variant = 'chrPAR1:2208957:C:G'
purple_variant = 'chrPAR1:2243088:C:G'

display(df_par1[df_par1.SNP.isin([green_variant, purple_variant])])

fig = manhattan_plot_PAR_with_highlights(
    merged_par=df_par1,
    title=f"{label} X Chromosome PAR1 Results",
    genomewide=5e-8,
    suggestive=1e-6,
    par_color="#4f77be",
    point_size=5,
    annotate_top_n=0,
    green_variant = green_variant,
    purple_variant = purple_variant
)

### Explore Top Variants

In [ ]:
# Genome-wide significant hits in the sex-combined results
display(df_nonpar[df_nonpar.P < 5e-8].sort_values(by='P', ascending=True).iloc[:10])
display(df_par[df_par.P < 5e-8].sort_values(by='P', ascending=True).iloc[:10])

In [ ]:
# Genome-wide significant hits in the sex-stratified non-PAR results
display(df_f_nonpar[df_f_nonpar.P < 5e-8].sort_values(by='P', ascending=True).iloc[:10])
display(df_m_nonpar[df_m_nonpar.P < 5e-8].sort_values(by='P', ascending=True).iloc[:10])

### QQ Plots

In [ ]:
# QQ Plots with all regions and sexes in one figure
plot_info = [
    (df_m_nonpar['P'], "Male Non-PAR"),
    (df_m_par['P'],    "Male PAR"),
    (df_f_nonpar['P'], "Female Non-PAR"),
    (df_f_par['P'],    "Female PAR")
]

fig, axes = plt.subplots(2, 2, figsize=(18, 12), facecolor="w")

for ax, (pdata, title) in zip(axes.ravel(), plot_info):
    qqplot(
        data=pdata,
        marker="o",
        title=title,
        xlabel=r"Expected -log(P)",
        ylabel=r"Observed -log(P)",
        ax=ax
    )

plt.tight_layout()
fig.savefig(f"{wd}/{label}_qq_plots.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# QQ Plots of the sex-combined results
plot_info = [
    (df_nonpar['P'], "Non-PAR"),
    (df_par['P'], "PAR")
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor="w")

for ax, (pdata, title) in zip(axes, plot_info):
    qqplot(
        data=pdata,
        marker="o",
        title=title,
        xlabel=r"Expected -log(P)",
        ylabel=r"Observed -log(P)",
        ax=ax
    )

plt.tight_layout()
fig.savefig(f"{wd}/{label}_qq_plots_merged.png", dpi=600, bbox_inches="tight")
plt.show()

### Genomic Inflation

In [ ]:
def lambda_gc_from_or_se(path, or_col="OR", se_col="LOG(OR)_SE", maf_thresh=0.03):
    df = pd.read_csv(path, sep="\t")
    df = df[df.A1_FREQ >= maf_thresh]

    # beta = log(OR)
    beta = np.log(df[or_col].astype(float))

    # Z = log(OR) / SE_log(OR)
    z = beta / df[se_col].astype(float)

    # chi-square with 1 df
    chisq = z**2

    # genomic inflation
    return np.median(chisq[np.isfinite(chisq)]) / 0.455


def neff_case_control(n_cases, n_controls):
    return 4 * n_cases * n_controls / (n_cases + n_controls)


def lambda_1000(lambda_gc, neff):
    return 1 + (lambda_gc - 1) * (1000 / neff)

In [ ]:
# Example usage -- case/control counts come from the PLINK log of the GWAS run
label = 'AJ'
region = 'PAR'
sex = 'female'

lam = lambda_gc_from_or_se(f'{wd}/working/R{rel}_prep/XWAS/{sex}/{region}/XWAS_{label}_{region}_{sex}.PHENO.glm.logistic.hybrid')

n_cases, n_controls = 491, 254
neff = neff_case_control(n_cases, n_controls)
lam1000 = lambda_1000(lam, neff)

print("lambda GC:", lam)
print("lambda_1000:", lam1000)